# Phase 13 — Attack-Family Transfer & Semantic Shift Analysis

## NetShield

**Purpose:** Identify which CICIDS2017 attack families transfer poorly after semantic alignment + CORAL/class weighting, and test whether their failures are associated with larger semantic feature-distribution shifts.

**Leakage rule:** CICIDS evaluation labels are used only for frozen error analysis. No CICIDS labels are used to fit preprocessing, CORAL, models, thresholds, or feature-selection decisions.


## Research question

> Are the remaining cross-dataset errors concentrated in attack families whose common semantic features differ more strongly between UNSW-NB15 and CICIDS2017?

This phase does **not** introduce another generic classifier. It directly tests the mechanism suggested by Phases 11–12: global alignment and class-prior handling help overall transfer, but some attack families remain difficult.


In [ ]:
from pathlib import Path
import json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_UNSW = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RAW_CICIDS = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
TABLE_DIR = PROJECT_ROOT / "outputs" / "attack_family_transfer_analysis" / "tables"
PLOT_DIR = PROJECT_ROOT / "outputs" / "attack_family_transfer_analysis" / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
COMMON = [
    "flow_duration_sec","fwd_packets","bwd_packets","fwd_bytes","bwd_bytes",
    "total_packets","total_bytes","packet_rate","fwd_packet_rate",
    "bwd_packet_rate","avg_packet_size"
]
print("Project root:", PROJECT_ROOT)


## 1. Load only the data required for semantic-shift analysis

The CICIDS loader uses chunks to avoid the RAM problem encountered in Phase 11.


In [ ]:
def load_unsw_common():
    train_files = sorted(RAW_UNSW.glob("*Training*.csv"))
    test_files = sorted(RAW_UNSW.glob("*Testing*.csv"))
    if not train_files or not test_files:
        raise FileNotFoundError("Could not find UNSW training/testing CSV files.")
    tr = pd.read_csv(train_files[0], low_memory=False)
    te = pd.read_csv(test_files[0], low_memory=False)
    df = pd.concat([tr, te], ignore_index=True)
    y = pd.to_numeric(df["label"], errors="coerce").astype("Int8")

    dur = pd.to_numeric(df["dur"], errors="coerce")
    fp = pd.to_numeric(df["spkts"], errors="coerce")
    bp = pd.to_numeric(df["dpkts"], errors="coerce")
    fb = pd.to_numeric(df["sbytes"], errors="coerce")
    bb = pd.to_numeric(df["dbytes"], errors="coerce")
    total_p = fp + bp
    total_b = fb + bb

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = dur
    x["fwd_packets"] = fp
    x["bwd_packets"] = bp
    x["fwd_bytes"] = fb
    x["bwd_bytes"] = bb
    x["total_packets"] = total_p
    x["total_bytes"] = total_b
    safe_dur = dur.replace(0, np.nan)
    safe_total_p = total_p.replace(0, np.nan)
    x["packet_rate"] = total_p / safe_dur
    x["fwd_packet_rate"] = fp / safe_dur
    x["bwd_packet_rate"] = bp / safe_dur
    x["avg_packet_size"] = total_b / safe_total_p
    return x, y


def load_cicids_common():
    needed = [
        "Flow Duration","Total Fwd Packets","Total Backward Packets",
        "Total Length of Fwd Packets","Total Length of Bwd Packets",
        "Flow Packets/s","Fwd Packets/s","Bwd Packets/s",
        "Average Packet Size","Label"
    ]
    frames = []

    for p in sorted(RAW_CICIDS.rglob("*.csv")):
        h = pd.read_csv(p, nrows=0)
        lookup = {
            str(c).strip().replace("\ufeff","").lower(): c
            for c in h.columns
        }
        sel = {c: lookup[c.lower()] for c in needed if c.lower() in lookup}

        if "Label" not in sel:
            print("skip", p.name)
            continue

        chunks = []
        for chunk in pd.read_csv(
            p,
            usecols=list(sel.values()),
            chunksize=100_000,
            low_memory=True
        ):
            chunk = chunk.rename(columns={v:k for k,v in sel.items()})
            for c in needed:
                if c != "Label" and c in chunk.columns:
                    chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
            chunk["Label"] = chunk["Label"].astype(str).str.strip()
            chunks.append(chunk)

        file_df = pd.concat(chunks, ignore_index=True)
        frames.append(file_df)
        print("loaded", p.name, len(file_df))
        del chunks, file_df
        gc.collect()

    if not frames:
        raise ValueError("No CICIDS CSV with Label found.")

    df = pd.concat(frames, ignore_index=True)

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = df["Flow Duration"] / 1e6
    x["fwd_packets"] = df["Total Fwd Packets"]
    x["bwd_packets"] = df["Total Backward Packets"]
    x["fwd_bytes"] = df["Total Length of Fwd Packets"]
    x["bwd_bytes"] = df["Total Length of Bwd Packets"]
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = df["Flow Packets/s"]
    x["fwd_packet_rate"] = df["Fwd Packets/s"]
    x["bwd_packet_rate"] = df["Bwd Packets/s"]
    x["avg_packet_size"] = df["Average Packet Size"]

    y = df["Label"].copy()
    return x, y


X_unsw, y_unsw = load_unsw_common()
X_cic, y_cic = load_cicids_common()

print("UNSW:", X_unsw.shape)
print("CICIDS:", X_cic.shape)


## 2. Clean only clearly invalid numeric values

We preserve genuine zeros and do not remove outliers merely because they are extreme. Non-finite values are treated as missing for distribution calculations.


In [ ]:
def clean_common(X):
    z = X.copy()
    z = z.replace([np.inf, -np.inf], np.nan)
    limit = np.finfo(np.float32).max
    for c in COMMON:
        z.loc[z[c].abs() > limit, c] = np.nan
    return z

X_unsw_c = clean_common(X_unsw)
X_cic_c = clean_common(X_cic)

print("Missing/non-finite cells — UNSW:", int(X_unsw_c.isna().sum().sum()))
print("Missing/non-finite cells — CICIDS:", int(X_cic_c.isna().sum().sum()))


## 3. Global semantic distribution shift

KS statistic is used as a descriptive effect-size measure. Because both datasets are large, p-values are not treated as the main evidence.


In [ ]:
rows = []

for f in COMMON:
    a = X_unsw_c[f].dropna().to_numpy()
    b = X_cic_c[f].dropna().to_numpy()
    stat = ks_2samp(a, b).statistic

    rows.append({
        "feature": f,
        "ks_statistic": stat,
        "unsw_median": np.median(a),
        "cicids_median": np.median(b),
        "unsw_iqr": np.percentile(a, 75) - np.percentile(a, 25),
        "cicids_iqr": np.percentile(b, 75) - np.percentile(b, 25),
        "unsw_zero_rate": float(np.mean(a == 0)),
        "cicids_zero_rate": float(np.mean(b == 0))
    })

global_shift = pd.DataFrame(rows).sort_values("ks_statistic", ascending=False)
display(global_shift)

global_shift.to_csv(
    TABLE_DIR / "01_global_semantic_shift.csv",
    index=False
)

plt.figure(figsize=(10,5))
plt.bar(global_shift["feature"], global_shift["ks_statistic"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("KS statistic")
plt.title("Global semantic distribution shift")
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_global_semantic_shift.png", dpi=180)
plt.show()


## 4. Load the frozen Phase 12 attack-family results

We do not retrain or tune a model in this phase. The Phase 12 attack-family results define the observed transfer changes that we will compare against semantic shifts.


In [ ]:
phase12_candidates = [
    PROJECT_ROOT / "outputs" / "class_prior_analysis" / "tables" / "05_attack_family_effect.csv",
    PROJECT_ROOT / "outputs" / "class_prior_analysis" / "tables" / "04_attack_family_effect.csv",
    PROJECT_ROOT / "outputs" / "class_prior_analysis" / "tables" / "03_attack_family_effect.csv"
]

attack_file = next((p for p in phase12_candidates if p.exists()), None)

if attack_file is None:
    raise FileNotFoundError(
        "Could not find the Phase 12 attack-family CSV. "
        "Place the Phase 12 output folder under outputs/class_prior_analysis/."
    )

attack_effect = pd.read_csv(attack_file)
print("Using:", attack_file)
display(attack_effect)


## 5. Define transfer groups from frozen evidence

Families with at least 300 examples are used for quantitative family-level comparisons. Tiny classes remain in the outputs but are treated cautiously.


In [ ]:
attack_effect["n"] = pd.to_numeric(attack_effect["n"], errors="coerce")
attack_effect["change"] = pd.to_numeric(attack_effect["change"], errors="coerce")

attack_effect["transfer_group"] = np.where(
    (attack_effect["n"] >= 300) & (attack_effect["change"] <= -0.20),
    "poor_transfer",
    np.where(
        (attack_effect["n"] >= 300) & (attack_effect["change"] >= 0.10),
        "improved_transfer",
        "other"
    )
)

display(
    attack_effect[
        ["attack_label","n","change","transfer_group"]
    ]
)

attack_effect.to_csv(
    TABLE_DIR / "02_transfer_groups.csv",
    index=False
)


## 6. Attack-family semantic shift profiles

For each CICIDS attack family, compare its semantic feature distribution against the UNSW **attack pool**.

**Important limitation:** UNSW and CICIDS attack taxonomies are not one-to-one. This is therefore not claiming that a CICIDS family corresponds exactly to a particular UNSW family. It asks whether each CICIDS family occupies a semantic region that differs from the overall UNSW attack traffic.


In [ ]:
unsw_attack = X_unsw_c.loc[y_unsw == 1]
cic_labels = y_cic.astype(str).str.strip()
cic_attack_mask = cic_labels.str.upper().ne("BENIGN")

profile_rows = []

for label in sorted(cic_labels[cic_attack_mask].unique()):
    mask = cic_labels.eq(label)
    target_family = X_cic_c.loc[mask]
    n = int(mask.sum())

    for f in COMMON:
        a = unsw_attack[f].dropna().to_numpy()
        b = target_family[f].dropna().to_numpy()

        ks = (
            ks_2samp(a, b).statistic
            if len(a) and len(b)
            else np.nan
        )

        profile_rows.append({
            "attack_label": label,
            "n": n,
            "feature": f,
            "ks_vs_unsw_attack": ks,
            "cicids_median": np.median(b) if len(b) else np.nan,
            "unsw_attack_median": np.median(a) if len(a) else np.nan
        })

family_feature_shift = pd.DataFrame(profile_rows)

family_feature_shift.to_csv(
    TABLE_DIR / "03_family_feature_shift.csv",
    index=False
)

display(family_feature_shift.head(25))


## 7. Summarize each attack family's semantic shift

The mean KS is a compact descriptive score across the 11 common features. The maximum KS identifies whether a family has at least one particularly large feature-level shift.


In [ ]:
family_shift = (
    family_feature_shift
    .groupby(["attack_label","n"], as_index=False)
    .agg(
        mean_ks=("ks_vs_unsw_attack","mean"),
        median_ks=("ks_vs_unsw_attack","median"),
        max_ks=("ks_vs_unsw_attack","max")
    )
    .sort_values("mean_ks", ascending=False)
)

family_shift = family_shift.merge(
    attack_effect[["attack_label","change","transfer_group"]],
    on="attack_label",
    how="left"
)

display(family_shift)

family_shift.to_csv(
    TABLE_DIR / "04_family_shift_summary.csv",
    index=False
)

plt.figure(figsize=(10,6))
plt.scatter(
    family_shift["mean_ks"],
    family_shift["change"],
    s=np.maximum(family_shift["n"], 10) / 5
)

for _, r in family_shift.iterrows():
    if r["n"] >= 300:
        plt.annotate(
            str(r["attack_label"])[:22],
            (r["mean_ks"], r["change"]),
            fontsize=8
        )

plt.axhline(0, linewidth=1)
plt.xlabel("Mean semantic KS shift vs UNSW attack pool")
plt.ylabel("Frozen detection-rate change")
plt.title("Attack-family transfer change vs semantic shift")
plt.tight_layout()
plt.savefig(
    PLOT_DIR / "02_family_shift_vs_transfer_change.png",
    dpi=180
)
plt.show()


## 8. Focus on the clearest poor-transfer and improved-transfer families


In [ ]:
eligible = family_shift[
    family_shift["n"] >= 300
].copy()

poor = (
    eligible
    .sort_values("change")
    .head(4)["attack_label"]
    .tolist()
)

improved = (
    eligible
    .sort_values("change", ascending=False)
    .head(4)["attack_label"]
    .tolist()
)

focus = list(dict.fromkeys(poor + improved))

print("Poor-transfer families:", poor)
print("Improved-transfer families:", improved)

focus_rows = family_feature_shift[
    family_feature_shift["attack_label"].isin(focus)
].copy()

focus_rows.to_csv(
    TABLE_DIR / "05_focus_family_feature_shift.csv",
    index=False
)

for label in focus:
    q = (
        focus_rows[
            focus_rows["attack_label"] == label
        ]
        .sort_values("ks_vs_unsw_attack", ascending=False)
    )

    plt.figure(figsize=(10,4.5))
    plt.bar(q["feature"], q["ks_vs_unsw_attack"])
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("KS vs UNSW attack pool")
    plt.title(f"Semantic shift profile: {label}")
    plt.tight_layout()

    safe = "".join(
        ch if ch.isalnum() else "_"
        for ch in str(label)
    )[:60]

    plt.savefig(
        PLOT_DIR / f"03_shift_{safe}.png",
        dpi=160
    )
    plt.show()


## 9. Final evidence summary

A negative association means families with larger semantic shifts tend to show larger detection-rate losses. This is **descriptive evidence, not causal proof**.


In [ ]:
quant = (
    family_shift[
        family_shift["n"] >= 300
    ]
    .dropna(subset=["mean_ks","change"])
    .copy()
)

if len(quant) >= 3:
    spearman_corr = quant["mean_ks"].corr(
        quant["change"],
        method="spearman"
    )
else:
    spearman_corr = np.nan

summary = {
    "n_attack_families_total": int(len(family_shift)),
    "n_attack_families_eligible_n_ge_300": int(len(quant)),
    "spearman_corr_mean_semantic_shift_vs_detection_change": (
        None if pd.isna(spearman_corr)
        else float(spearman_corr)
    ),
    "poor_transfer_families": poor,
    "improved_transfer_families": improved,
    "interpretation_rule": (
        "Negative correlation means larger semantic shift tends "
        "to accompany larger detection-rate loss. This is "
        "descriptive evidence, not causal proof."
    )
}

with open(
    TABLE_DIR / "06_phase13_evidence.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


# Interpretation guide

### Evidence supporting the hypothesis
Poor-transfer families show consistently larger semantic KS shifts than improved-transfer families, with a meaningfully negative family-level relationship.

### Mixed evidence
Some poor-transfer families have large shifts but others do not. This means semantic shift contributes, but attack-specific factors also matter.

### Weak/no relationship
Detection failure does not track semantic distribution shift. The remaining gap is then more likely related to attack taxonomy mismatch, discriminative feature differences, or other dataset-specific factors.

**Do not claim causality.** This phase is observational error analysis.
